In [ ]:
import pandas as pd

metadata = pd.read_csv(
    "../data/01_raw/Hall_2018/Hall_2018-metadata.csv"
)

print(metadata.head())
print(metadata.shape)
print(metadata["diabetes_type"].value_counts())

In [5]:
from pathlib import Path

glucose_dir = Path(
    "../data/01_raw/Hall_2018/Hall_2018-extracted-glucose-files"
)

glucose_files = list(glucose_dir.glob("*.csv"))

len(glucose_files)

57

In [ ]:
file_person_ids = {
    file.stem
    for file in glucose_files
}

metadata_person_ids = set(metadata["person_id"])

print("Metadata people:", len(metadata_person_ids))
print("Glucose files:", len(file_person_ids))

In [ ]:
# How many minutes normally separate consecutive CGM observations?
glucose = pd.read_csv(
    "../data/01_raw/Hall_2018/Hall_2018-extracted-glucose-files/1636-69-001.csv"
)

glucose["timestamp"] = pd.to_datetime(glucose["timestamp"])

glucose.head()

In [ ]:
# 
intervals = glucose["timestamp"].diff().dt.total_seconds() / 60

print(intervals.value_counts().sort_index())
print(intervals.describe())

In [ ]:
id="7e3r2k"
glucose["interval_minutes"] = (
    glucose["timestamp"].diff().dt.total_seconds() / 60
)

glucose[
    glucose["interval_minutes"] != 5
][
    ["timestamp", "glucose_value_mg_dl", "interval_minutes"]
]

In [ ]:
print(glucose["glucose_value_mg_dl"].describe())
print(glucose["glucose_value_mg_dl"].isna().sum())
print(glucose["glucose_value_mg_dl"].sort_values().head(10))
print(glucose["glucose_value_mg_dl"].sort_values().head(10))
print(glucose["glucose_value_mg_dl"].sort_values(ascending=False).head(10))

Plotting the data in matlplotlib to check the glucose over time for ECG mesuments

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

plt.plot(
    glucose["timestamp"],
    glucose["glucose_value_mg_dl"]
)

plt.axhline(
    180,
    linestyle="--"
)

plt.xlabel("Time")
plt.ylabel("Glucose (mg/dL)")
plt.title("CGM Glucose — Person 1636-69-001")

plt.show()

Gave us an erro due to the massive time gap

In [ ]:
glucose["interval_minutes"] = (
    glucose["timestamp"].diff().dt.total_seconds() / 60
)

glucose["segment_id"] = (
    glucose["interval_minutes"] > 15
).cumsum()

glucose.groupby("segment_id").agg(
    start_time=("timestamp", "min"),
    end_time=("timestamp", "max"),
    observations=("glucose_value_mg_dl", "count")
)